# Model Comparison - KHOTAA DFU Classification

This notebook provides comprehensive comparison of all trained models using their saved comprehensive metrics JSON files.

**Models Compared:**
- MobileNetV2
- ResNet101
- ResNet50
- EfficientNetV2-S
- GoogLeNet
- DenseNet

**Metrics Analyzed:**
- Cross-validation accuracy (mean ± std)
- Per-class metrics (precision, recall, F1, specificity, AUC)
- Confusion matrices
- Training efficiency (epochs, convergence)
- Model complexity vs performance

## 1. Setup and Load Results

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configuration
RESULTS_DIR = Path('models/classification/results')
OUTPUT_DIR = Path('models/classification/results/comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DPI = 300
FIGSIZE_LARGE = (16, 10)
FIGSIZE_MEDIUM = (12, 8)
FIGSIZE_SMALL = (10, 6)

print("Libraries loaded successfully")

In [ ]:
# Load all model results
def load_model_results(results_dir):
    """
    Load comprehensive metrics JSON files for all models.
    
    Returns:
        dict: Dictionary with model names as keys and results as values
    """
    models = {}
    results_path = Path(results_dir)
    
    # Expected model directories
    model_names = ['mobilenetv2', 'resnet101', 'resnet50', 'efficientnetv2s', 'googlenet', 'densenet']
    
    print("Loading model results...")
    for model_name in model_names:
        json_file = results_path / model_name / f'{model_name}_comprehensive_metrics.json'
        
        if json_file.exists():
            try:
                with open(json_file, 'r') as f:
                    models[model_name] = json.load(f)
                print(f"  Loaded: {model_name}")
            except Exception as e:
                print(f"  Error loading {model_name}: {e}")
        else:
            print(f"  Not found: {model_name} at {json_file}")
    
    print(f"\nTotal models loaded: {len(models)}")
    return models

# Load all results
models = load_model_results(RESULTS_DIR)

# Display loaded models
if models:
    print("\nLoaded Models:")
    for model_name in models.keys():
        print(f"  - {models[model_name]['model_info']['model_name']}")
else:
    print("\nNo models loaded. Please ensure comprehensive metrics JSON files exist.")

## 2. Cross-Validation Performance Comparison

In [ ]:
# Extract CV metrics
def extract_cv_metrics(models):
    """
    Extract cross-validation metrics from all models.
    
    Returns:
        pd.DataFrame: DataFrame with CV metrics for each model
    """
    cv_data = []
    
    for model_name, results in models.items():
        cv = results['cross_validation']
        cv_data.append({
            'Model': results['model_info']['model_name'],
            'Mean Accuracy': cv['mean_accuracy'] * 100,
            'Std Accuracy': cv['std_accuracy'] * 100,
            'Min Accuracy': cv['min_accuracy'] * 100,
            'Max Accuracy': cv['max_accuracy'] * 100,
            'Avg Epochs': cv['average_epochs'],
            'Best Fold': cv['best_fold']['fold_number'],
            'Best Fold Acc': cv['best_fold']['accuracy'] * 100
        })
    
    df = pd.DataFrame(cv_data)
    df = df.sort_values('Mean Accuracy', ascending=False).reset_index(drop=True)
    return df

cv_metrics = extract_cv_metrics(models)

# Display table
print("\n" + "="*80)
print("CROSS-VALIDATION PERFORMANCE COMPARISON")
print("="*80)
print(cv_metrics.to_string(index=False))
print("="*80)

# Export to CSV
output_path = OUTPUT_DIR / 'cv_comparison.csv'
cv_metrics.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

## 2.1. Comprehensive Metrics Table (All Models)

In [ ]:
# Create comprehensive metrics table for all models
def create_comprehensive_metrics_table(models):
    """
    Create a detailed table with all metrics for each model.
    
    Returns:
        pd.DataFrame: Comprehensive metrics table
    """
    all_metrics = []
    
    for model_name, results in models.items():
        model_display_name = results['model_info']['model_name']
        cv = results['cross_validation']
        val = results['validation_results']
        per_class = val['per_class_metrics']
        
        # Calculate average metrics across all classes
        avg_precision = np.mean([m['precision'] for m in per_class.values()])
        avg_recall = np.mean([m['recall'] for m in per_class.values()])
        avg_f1 = np.mean([m['f1_score'] for m in per_class.values()])
        avg_specificity = np.mean([m['specificity'] for m in per_class.values()])
        avg_sensitivity = np.mean([m['sensitivity'] for m in per_class.values()])
        avg_auc = np.mean([m['auc'] for m in per_class.values()])
        
        all_metrics.append({
            'Model': model_display_name,
            'CV Accuracy (%)': f"{cv['mean_accuracy']*100:.2f}",
            'CV Std (%)': f"{cv['std_accuracy']*100:.2f}",
            'Val Accuracy (%)': f"{val['accuracy']*100:.2f}",
            'Avg Precision': f"{avg_precision:.4f}",
            'Avg Recall': f"{avg_recall:.4f}",
            'Avg F1-Score': f"{avg_f1:.4f}",
            'Avg Specificity': f"{avg_specificity:.4f}",
            'Avg Sensitivity': f"{avg_sensitivity:.4f}",
            'Avg AUC': f"{avg_auc:.4f}",
            'Avg Epochs': f"{cv['average_epochs']:.1f}",
            'Best Fold': cv['best_fold']['fold_number'],
            'Best Fold Acc (%)': f"{cv['best_fold']['accuracy']*100:.2f}",
            # Sort key (numeric for sorting)
            '_sort_key': cv['mean_accuracy']
        })
    
    df = pd.DataFrame(all_metrics)
    # Sort by CV accuracy (descending)
    df = df.sort_values('_sort_key', ascending=False).reset_index(drop=True)
    # Remove sort key column
    df = df.drop('_sort_key', axis=1)
    # Add rank
    df.insert(0, 'Rank', range(1, len(df) + 1))
    
    return df

# Create comprehensive table
comprehensive_table = create_comprehensive_metrics_table(models)

# Display table
print("\n" + "="*150)
print("COMPREHENSIVE METRICS TABLE - ALL MODELS (Ordered by CV Accuracy)")
print("="*150)
print(comprehensive_table.to_string(index=False))
print("="*150)

# Export to CSV
output_path = OUTPUT_DIR / 'comprehensive_metrics_table.csv'
comprehensive_table.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

# Also display as styled pandas table for better readability
print("\n" + "="*150)
print("DETAILED VIEW:")
print("="*150)
for idx, row in comprehensive_table.iterrows():
    print(f"\n{row['Rank']}. {row['Model']}")
    print(f"   CV Accuracy: {row['CV Accuracy (%)']}% ± {row['CV Std (%)']}%")
    print(f"   Val Accuracy: {row['Val Accuracy (%)']}%")
    print(f"   Precision: {row['Avg Precision']} | Recall: {row['Avg Recall']} | F1: {row['Avg F1-Score']}")
    print(f"   Specificity: {row['Avg Specificity']} | Sensitivity: {row['Avg Sensitivity']} | AUC: {row['Avg AUC']}")
    print(f"   Avg Epochs: {row['Avg Epochs']} | Best Fold: {row['Best Fold']} ({row['Best Fold Acc (%)']}%)")
    print("   " + "-"*100)
print("="*150)

In [ ]:
# Visualization 1: Mean Accuracy with Error Bars
fig, ax = plt.subplots(figsize=FIGSIZE_MEDIUM)

models_sorted = cv_metrics.sort_values('Mean Accuracy', ascending=True)
y_pos = np.arange(len(models_sorted))

bars = ax.barh(y_pos, models_sorted['Mean Accuracy'], xerr=models_sorted['Std Accuracy'],
               alpha=0.8, capsize=5, error_kw={'linewidth': 2})

# Color bars by performance
colors = plt.cm.RdYlGn(models_sorted['Mean Accuracy'] / 100)
for bar, color in zip(bars, colors):
    bar.set_color(color)

ax.set_yticks(y_pos)
ax.set_yticklabels(models_sorted['Model'], fontsize=11)
ax.set_xlabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison (5-Fold CV)', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(models_sorted.iterrows()):
    ax.text(row['Mean Accuracy'] + row['Std Accuracy'] + 0.5, i, 
            f"{row['Mean Accuracy']:.2f}% ± {row['Std Accuracy']:.2f}%",
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
output_path = OUTPUT_DIR / 'model_comparison_accuracy.png'
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.show()

print(f"Saved to {output_path}")

In [ ]:
# Visualization 2: Box Plot of Fold Accuracies
fig, ax = plt.subplots(figsize=FIGSIZE_LARGE)

# Collect all fold accuracies
fold_data = []
model_labels = []

for model_name, results in models.items():
    model_display_name = results['model_info']['model_name']
    fold_accs = [fold['best_val_accuracy'] * 100 
                 for fold in results['cross_validation']['individual_folds']]
    fold_data.append(fold_accs)
    model_labels.append(model_display_name)

bp = ax.boxplot(fold_data, labels=model_labels, patch_artist=True, 
                showmeans=True, meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

# Color boxes
colors = plt.cm.Set3(np.linspace(0, 1, len(fold_data)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Fold Accuracies Across Models', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
output_path = OUTPUT_DIR / 'model_comparison_fold_distribution.png'
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.show()

print(f"Saved to {output_path}")

## 3. Per-Class Performance Comparison

In [ ]:
# Extract per-class metrics for all models
def extract_per_class_metrics(models):
    """
    Extract per-class metrics from all models.
    
    Returns:
        pd.DataFrame: DataFrame with per-class metrics
    """
    all_metrics = []
    
    for model_name, results in models.items():
        model_display_name = results['model_info']['model_name']
        per_class = results['validation_results']['per_class_metrics']
        
        for class_name, metrics in per_class.items():
            all_metrics.append({
                'Model': model_display_name,
                'Class': class_name,
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1-Score': metrics['f1_score'],
                'Specificity': metrics['specificity'],
                'Sensitivity': metrics['sensitivity'],
                'AUC': metrics['auc'],
                'Support': metrics['support']
            })
    
    return pd.DataFrame(all_metrics)

per_class_df = extract_per_class_metrics(models)

# Display summary statistics
print("\n" + "="*80)
print("PER-CLASS METRICS SUMMARY")
print("="*80)

for metric in ['Precision', 'Recall', 'F1-Score', 'AUC']:
    print(f"\n{metric}:")
    pivot = per_class_df.pivot_table(values=metric, index='Model', columns='Class', aggfunc='mean')
    print(pivot.to_string())
    print("-" * 80)

# Export to CSV
output_path = OUTPUT_DIR / 'per_class_comparison.csv'
per_class_df.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

In [ ]:
# Visualization 3: Heatmap of F1-Scores per Class
fig, ax = plt.subplots(figsize=FIGSIZE_MEDIUM)

pivot_f1 = per_class_df.pivot_table(values='F1-Score', index='Model', columns='Class', aggfunc='mean')

sns.heatmap(pivot_f1, annot=True, fmt='.3f', cmap='YlGnBu', 
            linewidths=0.5, cbar_kws={'label': 'F1-Score'}, ax=ax)

ax.set_title('F1-Score Comparison Across Models and Classes', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Model', fontsize=12, fontweight='bold')

plt.tight_layout()
output_path = OUTPUT_DIR / 'model_comparison_f1_heatmap.png'
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.show()

print(f"Saved to {output_path}")

In [ ]:
# Visualization 4: Grouped Bar Chart - Metrics by Class
classes = per_class_df['Class'].unique()
metrics_to_plot = ['Precision', 'Recall', 'F1-Score', 'AUC']

fig, axes = plt.subplots(2, 2, figsize=FIGSIZE_LARGE)
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    
    pivot = per_class_df.pivot_table(values=metric, index='Class', columns='Model', aggfunc='mean')
    pivot.plot(kind='bar', ax=ax, width=0.8)
    
    ax.set_title(f'{metric} by Class', fontsize=12, fontweight='bold')
    ax.set_xlabel('Class', fontsize=10, fontweight='bold')
    ax.set_ylabel(metric, fontsize=10, fontweight='bold')
    ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_ylim([0, 1.05])

plt.suptitle('Per-Class Performance Metrics Across All Models', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
output_path = OUTPUT_DIR / 'model_comparison_per_class_metrics.png'
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.show()

print(f"Saved to {output_path}")

## 4. Confusion Matrix Comparison

In [ ]:
# Visualization 5: All Confusion Matrices
n_models = len(models)
cols = 3
rows = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 6*rows))
axes = axes.flatten() if n_models > 1 else [axes]

for idx, (model_name, results) in enumerate(models.items()):
    ax = axes[idx]
    
    cm = np.array(results['validation_results']['confusion_matrix']['normalized'])
    labels = results['validation_results']['confusion_matrix']['labels']
    model_display_name = results['model_info']['model_name']
    
    sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues', 
                xticklabels=labels, yticklabels=labels, ax=ax,
                cbar_kws={'label': 'Percentage'})
    
    ax.set_title(f'{model_display_name}\nAcc: {results["validation_results"]["accuracy"]*100:.2f}%', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10, fontweight='bold')
    ax.set_ylabel('True', fontsize=10, fontweight='bold')

# Hide unused subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Normalized Confusion Matrices - All Models', 
             fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
output_path = OUTPUT_DIR / 'model_comparison_confusion_matrices.png'
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.show()

print(f"Saved to {output_path}")

## 5. Training Efficiency Analysis

In [ ]:
# Visualization 6: Training Efficiency (Accuracy vs Epochs)
fig, ax = plt.subplots(figsize=FIGSIZE_MEDIUM)

for model_name, results in models.items():
    model_display_name = results['model_info']['model_name']
    mean_acc = results['cross_validation']['mean_accuracy'] * 100
    avg_epochs = results['cross_validation']['average_epochs']
    std_acc = results['cross_validation']['std_accuracy'] * 100
    
    ax.errorbar(avg_epochs, mean_acc, yerr=std_acc, 
                marker='o', markersize=12, capsize=5, capthick=2,
                label=model_display_name, linewidth=0, elinewidth=2)
    
    # Add model name next to point
    ax.annotate(model_display_name, (avg_epochs, mean_acc), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.set_xlabel('Average Training Epochs', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Training Efficiency: Accuracy vs Training Time', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(alpha=0.3)
ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
output_path = OUTPUT_DIR / 'model_comparison_efficiency.png'
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.show()

print(f"Saved to {output_path}")

## 6. Statistical Comparison

In [ ]:
# Statistical significance testing (paired t-test)
print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTING (Paired t-test)")
print("="*80)

# Get all fold accuracies for each model
model_fold_accs = {}
for model_name, results in models.items():
    model_display_name = results['model_info']['model_name']
    fold_accs = [fold['best_val_accuracy'] 
                 for fold in results['cross_validation']['individual_folds']]
    model_fold_accs[model_display_name] = fold_accs

# Pairwise comparisons
model_names_list = list(model_fold_accs.keys())
comparison_results = []

for i in range(len(model_names_list)):
    for j in range(i+1, len(model_names_list)):
        model_a = model_names_list[i]
        model_b = model_names_list[j]
        
        accs_a = model_fold_accs[model_a]
        accs_b = model_fold_accs[model_b]
        
        # Paired t-test
        t_stat, p_value = stats.ttest_rel(accs_a, accs_b)
        
        mean_diff = (np.mean(accs_a) - np.mean(accs_b)) * 100
        
        significance = "***" if p_value < 0.001 else ("**" if p_value < 0.01 else ("*" if p_value < 0.05 else "ns"))
        
        comparison_results.append({
            'Model A': model_a,
            'Model B': model_b,
            'Mean Diff (%)': f"{mean_diff:+.2f}",
            't-statistic': f"{t_stat:.3f}",
            'p-value': f"{p_value:.4f}",
            'Significance': significance
        })

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values('p-value')

print("\nPairwise Comparisons:")
print(comparison_df.to_string(index=False))
print("\nSignificance levels: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print("="*80)

# Export
output_path = OUTPUT_DIR / 'statistical_comparison.csv'
comparison_df.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

## 7. Overall Rankings and Summary

In [ ]:
# Create comprehensive ranking
ranking_data = []

for model_name, results in models.items():
    model_display_name = results['model_info']['model_name']
    cv = results['cross_validation']
    val = results['validation_results']
    
    # Calculate average metrics across all classes
    per_class = val['per_class_metrics']
    avg_precision = np.mean([m['precision'] for m in per_class.values()])
    avg_recall = np.mean([m['recall'] for m in per_class.values()])
    avg_f1 = np.mean([m['f1_score'] for m in per_class.values()])
    avg_auc = np.mean([m['auc'] for m in per_class.values()])
    
    ranking_data.append({
        'Model': model_display_name,
        'CV Accuracy': cv['mean_accuracy'] * 100,
        'CV Std': cv['std_accuracy'] * 100,
        'Val Accuracy': val['accuracy'] * 100,
        'Avg Precision': avg_precision,
        'Avg Recall': avg_recall,
        'Avg F1': avg_f1,
        'Avg AUC': avg_auc,
        'Avg Epochs': cv['average_epochs']
    })

ranking_df = pd.DataFrame(ranking_data)
ranking_df = ranking_df.sort_values('CV Accuracy', ascending=False).reset_index(drop=True)
ranking_df.index = ranking_df.index + 1  # Start ranking from 1

print("\n" + "="*100)
print("OVERALL MODEL RANKINGS")
print("="*100)
print(ranking_df.to_string())
print("="*100)

# Export
output_path = OUTPUT_DIR / 'overall_ranking.csv'
ranking_df.to_csv(output_path)
print(f"\nSaved to {output_path}")

In [ ]:
# Visualization 7: Radar Chart for Top 3 Models
from math import pi

# Select top 3 models
top_3 = ranking_df.head(3)

# Metrics for radar chart
categories = ['CV Accuracy', 'Avg Precision', 'Avg Recall', 'Avg F1', 'Avg AUC']

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

# Number of variables
num_vars = len(categories)
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

# Plot for each model
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, (_, row) in enumerate(top_3.iterrows()):
    values = [
        row['CV Accuracy'] / 100,  # Normalize to 0-1
        row['Avg Precision'],
        row['Avg Recall'],
        row['Avg F1'],
        row['Avg AUC']
    ]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=colors[idx])
    ax.fill(angles, values, alpha=0.15, color=colors[idx])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11, fontweight='bold')

# Set y-axis limits
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], size=9)

ax.set_title('Top 3 Models - Performance Radar Chart', 
             fontsize=14, fontweight='bold', pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True)

plt.tight_layout()
output_path = OUTPUT_DIR / 'model_comparison_radar.png'
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.show()

print(f"Saved to {output_path}")

## 8. Generate Comprehensive Report

In [ ]:
# Generate markdown report
report = f"""# Model Comparison Report - KHOTAA DFU Classification

**Generated:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## Executive Summary

This report presents a comprehensive comparison of {len(models)} deep learning models trained for Diabetic Foot Ulcer (DFU) classification.

### Models Evaluated
{chr(10).join([f"- {results['model_info']['model_name']}" for results in models.values()])}

### Dataset Information
- **Total Samples:** {list(models.values())[0]['dataset_info']['total_samples']}
- **Training Samples:** {list(models.values())[0]['dataset_info']['training_samples']}
- **Test Samples:** {list(models.values())[0]['dataset_info']['test_samples']}
- **Classes:** {', '.join(list(models.values())[0]['dataset_info']['class_distribution'].keys())}
- **Cross-Validation:** {list(models.values())[0]['dataset_info']['num_folds']}-Fold Stratified

## Top Performing Models

###  1st Place: {ranking_df.iloc[0]['Model']}
- **CV Accuracy:** {ranking_df.iloc[0]['CV Accuracy']:.2f}% ± {ranking_df.iloc[0]['CV Std']:.2f}%
- **Avg F1-Score:** {ranking_df.iloc[0]['Avg F1']:.4f}
- **Avg AUC:** {ranking_df.iloc[0]['Avg AUC']:.4f}
- **Training Epochs:** {ranking_df.iloc[0]['Avg Epochs']:.1f}

###  2nd Place: {ranking_df.iloc[1]['Model']}
- **CV Accuracy:** {ranking_df.iloc[1]['CV Accuracy']:.2f}% ± {ranking_df.iloc[1]['CV Std']:.2f}%
- **Avg F1-Score:** {ranking_df.iloc[1]['Avg F1']:.4f}
- **Avg AUC:** {ranking_df.iloc[1]['Avg AUC']:.4f}
- **Training Epochs:** {ranking_df.iloc[1]['Avg Epochs']:.1f}

###  3rd Place: {ranking_df.iloc[2]['Model']}
- **CV Accuracy:** {ranking_df.iloc[2]['CV Accuracy']:.2f}% ± {ranking_df.iloc[2]['CV Std']:.2f}%
- **Avg F1-Score:** {ranking_df.iloc[2]['Avg F1']:.4f}
- **Avg AUC:** {ranking_df.iloc[2]['Avg AUC']:.4f}
- **Training Epochs:** {ranking_df.iloc[2]['Avg Epochs']:.1f}

## Key Findings

1. **Best Overall Performance:** {ranking_df.iloc[0]['Model']} achieved the highest mean accuracy of {ranking_df.iloc[0]['CV Accuracy']:.2f}%

2. **Most Consistent:** The model with lowest standard deviation is {ranking_df.sort_values('CV Std').iloc[0]['Model']} ({ranking_df.sort_values('CV Std').iloc[0]['CV Std']:.2f}%)

3. **Most Efficient:** {ranking_df.sort_values('Avg Epochs').iloc[0]['Model']} converged fastest with {ranking_df.sort_values('Avg Epochs').iloc[0]['Avg Epochs']:.1f} epochs on average

4. **Best AUC:** {ranking_df.sort_values('Avg AUC', ascending=False).iloc[0]['Model']} achieved the highest average AUC of {ranking_df.sort_values('Avg AUC', ascending=False).iloc[0]['Avg AUC']:.4f}

## Recommendations

Based on the comprehensive analysis:

- **For Production:** Use {ranking_df.iloc[0]['Model']} for best overall performance
- **For Resource-Constrained Environments:** Consider {ranking_df.sort_values('Avg Epochs').iloc[0]['Model']} for faster training
- **For Reliability:** {ranking_df.sort_values('CV Std').iloc[0]['Model']} offers most consistent predictions

## Generated Files

- `cv_comparison.csv` - Cross-validation metrics comparison
- `per_class_comparison.csv` - Detailed per-class metrics
- `statistical_comparison.csv` - Statistical significance tests
- `overall_ranking.csv` - Overall model rankings
- Multiple visualization PNG files

---
*Report generated automatically from comprehensive metrics JSON files*
"""

# Save report
output_path = OUTPUT_DIR / 'MODEL_COMPARISON_REPORT.md'
with open(output_path, 'w') as f:
    f.write(report)

print("\n" + "="*80)
print("REPORT GENERATED SUCCESSFULLY")
print("="*80)
print(f"\nSaved to {output_path}")
print("\nGenerated Files:")
print("  - cv_comparison.csv")
print("  - per_class_comparison.csv")
print("  - statistical_comparison.csv")
print("  - overall_ranking.csv")
print("  - MODEL_COMPARISON_REPORT.md")
print("  - Multiple visualization PNG files")
print("\n" + "="*80)